# Image and Time Series Classification with the SIGN Rule

This tutorial showcases the LRP **SIGN** rule, which redistributes the
relevance of a layer by the sign of its input rather than by the input itself,
and thereby reduces the magnitude-related bias which may distort explanations for time series data and images with very high or low contrast. It is
intended for the first layer of a model, and was introduced in [Gumpfer et
al. (2023)](https://doi.org/10.1016/j.inffus.2023.101883) for image data. We will use it on two classifiers:
VGG16 [(Simonyan & Zisserman, 2015)](https://arxiv.org/abs/1409.1556) trained on
ImageNet, and a 1D-CNN which detects ischemia in 12-lead ECGs [(Gumpfer et al.,
2024)](https://doi.org/10.1007/978-3-031-66535-6_36), whose trained weights are
published in the accompanying
[AIME2024](https://github.com/nilsgumpfer/AIME2024) repository. The ECGs are
taken from the PTB-XL dataset [(Wagner et al.,
2020)](https://doi.org/10.1038/s41597-020-0495-6), which is published on
[PhysioNet](https://physionet.org/content/ptb-xl/).

## Table of Contents
* [1. Preparation](<#1.-Preparation>)
* [2. Image Classification with VGG16](<#2.-Image-Classification-with-VGG16>)
    * [2.1 Preparing the Image and the Model](<#2.1-Preparing-the-Image-and-the-Model>)
    * [2.2 LRP with Epsilon and EpsilonSIGN](<#2.2-LRP-with-Epsilon-and-EpsilonSIGN>)
    * [2.3 Visualizing the Attributions](<#2.3-Visualizing-the-Attributions>)
* [3. Time Series Classification with a 1D-CNN](<#3.-Time-Series-Classification-with-a-1D-CNN>)
    * [3.1 Preparing the ECG](<#3.1-Preparing-the-ECG>)
    * [3.2 Preparing the Model](<#3.2-Preparing-the-Model>)
    * [3.3 LRP with Epsilon and EpsilonSIGN](<#3.3-LRP-with-Epsilon-and-EpsilonSIGN>)
    * [3.4 Visualizing the Attributions](<#3.4-Visualizing-the-Attributions>)

## 1. Preparation

First, we install **Zennit**. This includes its dependencies `Pillow`, `torch`
and `torchvision`. The time series part of this tutorial additionally needs
`h5py`, `matplotlib`, `scipy` and `wfdb`:

In [ ]:
%pip install zennit h5py matplotlib scipy wfdb

Then, we import necessary modules, classes and functions:

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
import wfdb
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from PIL import Image
from scipy.signal import butter, filtfilt, iirnotch
from torch import nn
from torchvision.models import vgg16, VGG16_Weights
from torchvision.transforms import Compose, Resize, ToTensor, Normalize as TorchNormalize

from zennit.attribution import Gradient
from zennit.composites import EpsilonSIGNComposite, LayerMapComposite, layer_map_base
from zennit.rules import Epsilon
from zennit.types import Convolution

plt.rcParams.update({'font.family': 'serif', 'figure.dpi': 200})

Both examples compare the `SIGN` rule against the `Epsilon` rule in the first
layer, so we need two **Composites**. Zennit provides `EpsilonSIGNComposite`,
which uses the `SIGN` rule for the first linear (dense, convolutional) layer and
the `Epsilon` rule for all other linear layers. The reference, which uses the
`Epsilon` rule in every layer, we build ourselves with the abstract
`LayerMapComposite`, where `layer_map_base` supplies the mapping for the
activations and the pooling layers:

In [ ]:
class EpsilonComposite(LayerMapComposite):
    """LRP-epsilon for every conv and linear layer."""
    def __init__(self, epsilon=1e-6, stabilizer=1e-6):
        super().__init__(layer_map=layer_map_base(stabilizer) + [
            (Convolution, Epsilon(epsilon=epsilon)),
            (nn.Linear, Epsilon(epsilon=epsilon)),
        ])

## 2. Image Classification with VGG16

We start with the image classifier, VGG16 [(Simonyan & Zisserman,
2015)](https://arxiv.org/abs/1409.1556). Feel free to replace it with any other
version of VGG.

### 2.1 Preparing the Image and the Model
We download a photo of a [North African
ostrich](https://commons.wikimedia.org/wiki/File:North_african_ostrich_(Struthio_camelus_camelus)_in_Morocco.jpg)
from Wikimedia Commons:

In [ ]:
torch.hub.download_url_to_file(
    'https://upload.wikimedia.org/wikipedia/commons/4/42/'
    'North_african_ostrich_%28Struthio_camelus_camelus%29_in_Morocco.jpg',
    'ostrich.jpg',
)

We load and prepare the data. The image is resized such that the shorter side is
224 pixels in size, converted to a `torch.Tensor`, and then normalized according
to the channel-wise mean and standard deviation of the ImageNet dataset. The
normalization centers the input of the first layer around zero, which is what
the separation threshold `mu` of the `SIGN` rule refers to, so `mu=0` is the
appropriate choice here:

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

# define the full tensor transform
transform = Compose([Resize(224), ToTensor(), TorchNormalize(IMAGENET_MEAN, IMAGENET_STD)])

# load the image
image = Image.open('ostrich.jpg').convert('RGB')

# transform the PIL image and insert a batch-dimension
data = transform(image)[None]

We load the model, and use the class it predicts as the target of the
attribution:

In [ ]:
# load the model and set it to evaluation mode
weights = VGG16_Weights.IMAGENET1K_V1
model = vgg16(weights=weights).eval()

# compute the model output
output = model(data)
pred = output.argmax(1).item()
label = weights.meta['categories'][pred]
prob = output.softmax(1)[0, pred].item()

# choose the predicted class as the target for the attribution
target = torch.eye(1000)[[pred]]

print('Predicted: {} (p = {:.3f})'.format(label, prob))

### 2.2 LRP with Epsilon and EpsilonSIGN
We compute the LRP-attribution for both **Composites** using the `Gradient`
**Attributor**, sum the relevance over the color channels, and normalize it to
`[-1, 1]`:

In [ ]:
mu = 0
epsilon = 0.2

composites = {
    'LRP Epsilon': EpsilonComposite(epsilon=epsilon),
    'LRP Epsilon / SIGN': EpsilonSIGNComposite(mu=mu, epsilon=epsilon),
}

relevances = {}
for name, composite in composites.items():
    # create the attributor, specifying model and composite
    with Gradient(model=model, composite=composite) as attributor:
        # compute the model output and attribution
        _, attribution = attributor(data, target)
    r = np.nan_to_num(attribution.detach().numpy()).sum(1)[0]  # sum over color channels
    relevances[name] = r / np.abs(r).max()                     # normalize to [-1, 1]

### 2.3 Visualizing the Attributions
We show the image next to the attributions, so that they can be compared
side-by-side. Red means positive relevance, blue means negative:

In [ ]:
# the image as the network sees it (undo the normalization) so it lines up with the heatmaps
shown = np.clip(data[0].permute(1, 2, 0).numpy() * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

fig, axs = plt.subplots(1, 1 + len(relevances), figsize=(2.4 * (1 + len(relevances)), 3))
axs[0].imshow(shown)
axs[0].set_title('Image', fontsize=9)
for ax, (name, r) in zip(axs[1:], relevances.items()):
    im = ax.imshow(r, cmap='seismic', clim=(-1, 1))
    ax.set_title(name, fontsize=9)
for ax in axs:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('VGG16 — {} (p = {:.2f})'.format(label, prob))
cbar = fig.colorbar(ScalarMappable(Normalize(-1, 1), 'seismic'), ax=axs, shrink=0.6, aspect=30, pad=0.04)
cbar.set_label('relevance (normalized)', fontsize=8)
plt.show()

## 3. Time Series Classification with a 1D-CNN

The **SIGN** rule was originally proposed for time series data, where the sign
of the signal carries the meaning and its magnitude around the baseline is
small. We use a 1D-CNN which detects ischemia (ISCH) from a 12-lead ECG, and one
record of the PTB-XL dataset [(Wagner et al.,
2020)](https://doi.org/10.1038/s41597-020-0495-6). Both the
model and its trained weights are those of [Gumpfer et al.
(2024)](https://doi.org/10.1007/978-3-031-66535-6_36), who compared XAI methods
for ECG interpretation on four cardiac pathologies.

### 3.1 Preparing the ECG
We download a single record, which consists of a header file and the raw signal:

In [ ]:
torch.hub.download_url_to_file(
    'https://physionet.org/files/ptb-xl/1.0.3/records500/12000/12131_hr.hea',
    '12131_hr.hea',
)
torch.hub.download_url_to_file(
    'https://physionet.org/files/ptb-xl/1.0.3/records500/12000/12131_hr.dat',
    '12131_hr.dat',
)

Raw ECGs contain baseline wander, power-line hum and high-frequency noise, all
of which would end up in the attribution. We remove them with the same filters
which were used to train the model, based on
[paulvangentcom/heartrate_analysis_python](https://github.com/paulvangentcom/heartrate_analysis_python):

In [ ]:
LEAD_INDEX = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


# ---- signal filtering (based on paulvangentcom/heartrate_analysis_python) ----
def filter_signal(data, cutoff, fs, filtertype, order=2):
    nyq = 0.5 * fs
    if filtertype == 'lowpass':
        b, a = butter(order, cutoff / nyq, btype='low')
    elif filtertype == 'notch':
        b, a = iirnotch(cutoff, Q=0.05, fs=fs)
    return filtfilt(b, a, data)


def adjust_baseline(lead, fs):
    """Shift the lead so that its flattest 200 ms window sits at zero."""
    w, step = int(fs * 0.2), int(fs * 0.1)
    best, adjustment = np.inf, 0
    for i in range(0, len(lead) - w, step):
        window = lead[i:i + w]
        spread = (np.max(window) - np.min(window)) ** 2
        if spread < best:
            best, adjustment = spread, np.mean(window)
    return lead - adjustment


def filter_lead(lead, fs):
    lead = filter_signal(lead, 0.05, fs, 'notch')   # baseline wander removal
    lead = adjust_baseline(lead, fs)                # baseline adjustment
    lead = filter_signal(lead, 50, fs, 'notch')     # 50 Hz power-line
    lead = filter_signal(lead, 40, fs, 'lowpass')   # 40 Hz low-pass
    return lead


def load_and_preprocess_ecg(record_id, subsample_start=0, window=2000, fs=500, src_dir='data'):
    """Load a WFDB record, filter every lead, return a (time, leads) window."""
    signal, _ = wfdb.rdsamp('{}/{}'.format(src_dir, record_id))  # (time, leads)
    ecg = np.nan_to_num(signal).T                               # (leads, time)
    ecg = np.array([filter_lead(lead, fs) for lead in ecg])
    ecg = ecg[:, subsample_start:subsample_start + window]
    return ecg.T                                               # (time, leads)

We load the record, filter every lead and cut out a window of 4 seconds:

In [ ]:
ecg = load_and_preprocess_ecg('12131_hr', subsample_start=200, src_dir='.')

# insert a batch-dimension; the model expects (batch, leads, time)
data_ecg = torch.from_numpy(ecg.T[None]).float()

### 3.2 Preparing the Model
The model is the original Keras 1D-CNN of [Gumpfer et al.
(2024)](https://doi.org/10.1007/978-3-031-66535-6_36), ported to PyTorch and
loaded straight from the Keras `weights.h5`, so no TensorFlow is needed:

In [ ]:
class ECGConvNet(nn.Module):
    """PyTorch port of the AIME2024 Keras 1D-CNN.

    5x [Conv1D(64, 3, same) -> ELU -> MaxPool1D(2)], then global average pooling and
    3x [Dense(64) -> ELU] plus a final Dense. The softmax is left off on purpose:
    LRP is applied to the logits. PyTorch expects (batch, leads, time).
    """
    def __init__(self, num_leads=12, num_classes=2, filters=64, units=64):
        super().__init__()
        conv, c_in = [], num_leads
        for _ in range(5):
            conv += [nn.Conv1d(c_in, filters, 3, padding=1), nn.ELU(), nn.MaxPool1d(2)]
            c_in = filters
        self.features = nn.Sequential(*conv)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        dense, f_in = [], filters
        for _ in range(3):
            dense += [nn.Linear(f_in, units), nn.ELU()]
            f_in = units
        dense += [nn.Linear(f_in, num_classes)]
        self.classifier = nn.Sequential(*dense)

    def forward(self, x):
        return self.classifier(self.flatten(self.pool(self.features(x))))


def load_ecg_model(weightspath):
    """Build the model and copy the Keras weights.h5 into it, transposing to PyTorch layout."""
    model = ECGConvNet()
    weights = {}
    with h5py.File(weightspath, 'r') as f:
        f.visititems(lambda name, obj: weights.setdefault(name.split('/')[0], {}).update(
            {name.split('/')[-1].split(':')[0]: np.array(obj)}) if isinstance(obj, h5py.Dataset) else None)

    convs = [m for m in model.features if isinstance(m, nn.Conv1d)]
    denses = [m for m in model.classifier if isinstance(m, nn.Linear)]
    conv_names = ['conv1d'] + ['conv1d_{}'.format(i) for i in range(1, len(convs))]
    dense_names = ['dense'] + ['dense_{}'.format(i) for i in range(1, len(denses))]

    with torch.no_grad():
        for layer, name in zip(convs, conv_names):  # Keras (k, in, out) -> torch (out, in, k)
            layer.weight.copy_(torch.from_numpy(weights[name]['kernel'].transpose(2, 1, 0).copy()))
            layer.bias.copy_(torch.from_numpy(weights[name]['bias']))
        for layer, name in zip(denses, dense_names):  # Keras (in, out) -> torch (out, in)
            layer.weight.copy_(torch.from_numpy(weights[name]['kernel'].T.copy()))
            layer.bias.copy_(torch.from_numpy(weights[name]['bias']))
    return model.eval()

We download the trained weights and load them into the model. The ischemia class
is the second output, which we use as the target of the attribution:

In [ ]:
torch.hub.download_url_to_file(
    'https://github.com/nilsgumpfer/AIME2024/raw/main/models/ISCH/weights.h5',
    'isch-weights.h5',
)

# load the model and set it to evaluation mode
model_ecg = load_ecg_model('isch-weights.h5')

# compute the model output
pathology_class = 1
output = model_ecg(data_ecg)
prob = output.softmax(1)[0, pathology_class].item()

# choose the ischemia class as the target for the attribution
target_ecg = torch.eye(output.shape[1])[[pathology_class]]

print('{} / {}: p = {:.3f}'.format('ISCH', '12131_hr', prob))

### 3.3 LRP with Epsilon and EpsilonSIGN
We again compute the LRP-attribution for both **Composites**. As in the original
work, we keep only the positive relevance, normalize it, threshold it, and
amplify it for visibility:

In [ ]:
mu = 0
epsilon = 0.001
posthresh = 0.2
cmap_adjust = 0.3

composites = {
    'LRP Epsilon': EpsilonComposite(epsilon=epsilon),
    'LRP Epsilon / SIGN': EpsilonSIGNComposite(mu=mu, epsilon=epsilon),
}

explanations = {}
for name, composite in composites.items():
    # create the attributor, specifying model and composite
    with Gradient(model=model_ecg, composite=composite) as attributor:
        # compute the model output and attribution
        _, attribution = attributor(data_ecg, target_ecg)
    r = np.nan_to_num(attribution.detach().numpy()[0].T)  # (time, leads)
    r[r < 0] = 0                                          # positives only
    r = r / np.abs(r).max()                              # normalize
    r[r <= posthresh] = 0                                # threshold
    r[r > posthresh] += cmap_adjust                      # amplify for visibility
    explanations[name] = r

### 3.4 Visualizing the Attributions
A heatmap is not a useful visualization for a signal, so we instead draw the
relevance as bubbles on top of the trace, where both the size and the color show
how relevant a sample is. The leads are stacked on top of each other, on a grid
which imitates ECG paper:

In [ ]:
RELEVANCE_CMAP = LinearSegmentedColormap.from_list('FairReds', [(1, 1, 1), (1, 0, 0)])


def plot_ecg_panel(ax, ecg, relevance, title, show_labels, fs=500, lead_spacing=2.8, y_margin=1.0, bubble=20):
    """Draw all 12 leads of one ECG stacked on a single axes, with relevance bubbles on the trace."""
    ecg, relevance = ecg.T, relevance.T  # (leads, time)
    n_leads, n = ecg.shape
    secs = n / fs
    t = np.arange(n) / fs
    y_min, y_max = -(n_leads - 1) * lead_spacing - y_margin, y_margin

    # ECG paper grid (0.2 s / 0.5 mV major, 5 minor subdivisions)
    ax.set_xticks(np.arange(0, secs + 1e-9, 0.2))
    ax.set_yticks(np.arange(np.ceil(y_min / 0.5) * 0.5, y_max, 0.5))
    ax.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))
    ax.grid(which='major', linewidth=0.4, color=(0.98, 0.82, 0.82))
    ax.grid(which='minor', linewidth=0.25, color=(0.99, 0.91, 0.91))
    ax.set_axisbelow(True)
    ax.set_xlim(0, secs)
    ax.set_ylim(y_min, y_max)
    ax.tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)

    for lead in range(n_leads):
        y = ecg[lead] - lead * lead_spacing
        ax.plot(t, y, linewidth=0.7, color=(0.1, 0.1, 0.1), zorder=3)
        if show_labels:
            ax.text(-0.015, -lead * lead_spacing + 0.45, LEAD_INDEX[lead], fontsize=7.5, ha='right',
                    va='center', transform=ax.get_yaxis_transform(), clip_on=False)
        mask = relevance[lead] > 0
        if mask.any():
            z = relevance[lead][mask]
            order = np.argsort(z)  # strongest relevance drawn last
            ax.scatter(t[mask][order], y[mask][order], c=z[order], cmap=RELEVANCE_CMAP, s=z[order] * bubble,
                       vmin=0, vmax=1, zorder=2, linewidths=0)
    ax.set_xlabel('{:.0f} s, 25 mm/s, 10 mm/mV'.format(secs), fontsize=7, color=(0.35, 0.35, 0.35))
    ax.set_title(title, fontsize=9)


def plot_ecg_comparison(ecg, explanations, suptitle):
    """One panel per XAI method, all showing the same ECG, for side-by-side comparison."""
    fig, axs = plt.subplots(1, len(explanations), figsize=(3.4 * len(explanations), 10.5), sharey=True)
    for i, (ax, (name, relevance)) in enumerate(zip(axs, explanations.items())):
        plot_ecg_panel(ax, ecg, relevance, title=name, show_labels=(i == 0))
    fig.suptitle(suptitle, fontsize=13, y=0.99)
    cbar = fig.colorbar(ScalarMappable(Normalize(0, 1), RELEVANCE_CMAP), ax=axs, shrink=0.3,
                        aspect=30, pad=0.01)
    cbar.set_label('relevance (normalized, positives only)', fontsize=8)
    plt.show()

Show the two attributions side-by-side:

In [ ]:
plot_ecg_comparison(ecg, explanations, suptitle='{} — {} (p = {:.2f})'.format('12131_hr', 'ISCH', prob))